# Advanced Company Brochure Maker

**LLM Engineering, Assignment 1.** A source-based information kit about two companies in the same field:
four Hebrew brochures (potential customers and job candidates for each company), a comparison table,
and a baseline brochure for the prompt experiment.

This notebook extends the class notebook `day5.ipynb` (collect links → choose with an LLM → collect content →
write a brochure → stream it). The logic lives in the `brochure/` package; this notebook runs it step by step,
shows the results, and holds the manual review.

![Architecture](docs/architecture.svg)

**How to run:** fill in the two companies in section 2, then run the cells in order.
A full run makes about 8 model calls: 2 link selections, 4 brochures, 1 comparison and 1 baseline.
Section 10 re-runs all checks from the saved files of a run, with no model calls.

## 1. Setup

In [11]:
from IPython.display import HTML, Markdown, display

from brochure import audit, config
from brochure.checks import check_saved_output, format_checks
from brochure.config import CONTEXT_CHAR_BUDGET, MODEL, Company, check_api_key, get_companies
from brochure.context import build_context, format_report
from brochure.generate import generate_brochure, generate_comparison, rtl_markdown
from brochure.links import build_candidates, save_candidates, select_links
from brochure.runlog import RunLog
from brochure.sources import (add_replacement_source, collect_sources, fetch_home, load_sources,
                              save_sources, summarize_sources)

check_api_key()
print(f"Model: {MODEL} | context budget: {CONTEXT_CHAR_BUDGET:,} characters of source text per call")

API key found and looks like a Claude API key.
Model: claude-opus-5-5 | context budget: 12,000 characters of source text per call


## 2. The two companies

Choose two companies **in the same field** whose sites can be read by the class scraper
(`scraper.py`). For each company we use the home page and up to 3 more pages, with no recursive crawling.

- `company_id`: a short slug used in file names (e.g. `"monday"`)
- `name`: the display name
- `url`: the home page

In [12]:
COMPANY_A = Company(company_id="ivory", name="אייבורי", url="https://www.ivory.co.il", letter="A")
COMPANY_B = Company(company_id="lastprice", name="לאסטפרייס", url="https://www.lastprice.co.il", letter="B")
FIELD = "קמעונאות מחשבים ואלקטרוניקה"

config.COMPANIES[:] = [COMPANY_A, COMPANY_B]
companies = get_companies()
assert all("TODO" not in c.company_id + c.name + c.url for c in companies), "Fill in both companies first."
for c in companies:
    print(f"{c.letter}: {c.name} ({c.url})  id={c.company_id}")
print("Field:", FIELD)

A: אייבורי (https://www.ivory.co.il)  id=ivory
B: לאסטפרייס (https://www.lastprice.co.il)  id=lastprice
Field: קמעונאות מחשבים ואלקטרוניקה


**Site choice notes.** Why these two companies, and the sites that were tried and replaced
(full details in `docs/attempts_log.md`):

| Company | Site | Readable by the scraper? | Notes |
|---|---|---|---|
| A: אייבורי | `www.ivory.co.il` | Yes: HTTP 200, 69,006 chars, 721 candidate links | A computer, mobile and electronics retail chain with physical stores. |
| B: לאסטפרייס | `www.lastprice.co.il` | Yes: HTTP 200, 15,836 chars, 487 candidate links | An online electronics and home-goods retailer, in the same field. Chosen to replace מחסני חשמל. |
| Replaced: מחסני חשמל | `www.payngo.co.il` | **No: HTTP 403** | The server blocks the class scraper, so no text could be collected. Run `20261127-163931-b1fd` stopped at the home page (1 failed request, 0 model calls). |
| Also checked, not used | `ksp.co.il`, `www.shekem-electric.co.il` | No: HTTP 403 | Blocked like payngo. |
| Also checked, not used | `www.machsanei-hashmal.co.il` | No: the domain does not resolve | |
| Also checked, not used | `www.bug.co.il` | Yes: HTTP 200, 28,935 chars | Readable alternative, not needed. |
| Also checked, not used | `www.electra.co.il` | Yes, but only 3,916 chars | A manufacturing group rather than a retail chain. |

## 3. Start a run

Everything from this run is saved under `runs/<run_id>/`: candidates, link selections, sources,
exact contexts and prompts, outputs with their metadata and checks, and the log (`log.jsonl`).

In [4]:
run = RunLog()
print("Run ID:", run.run_id)
print("Folder:", run.dir)

Run ID: 20261128-101549-d492
Folder: C:\Users\User\Documents\cursor\LLL_HM\LLM_HW1_Brochure_Maker\runs\20261128-101549-d492


## 4. Links and sources (per company)

**4a. Candidate links, built in code.** The home page is fetched once. Its links are resolved, `#fragments` are
removed, `mailto:` / `tel:` / `javascript:` and links to other sites are dropped (`www.` and the bare domain
count as the same site), the home page, privacy / terms pages and file links are dropped, and duplicates are
removed. The resulting list is saved, since it is exactly what the model receives.

In [4]:
homes, candidates, selections, collections = {}, {}, {}, {}

for co in companies:
    home = fetch_home(co, run=run)
    homes[co.company_id] = home
    print(f"{co.name}: HTTP {home['http_status']}, final URL {home['final_url']}, "
          f"{len(home['text']):,} chars of text, error: {home['error']}")
    assert home["error"] is None, f"The home page of {co.name} could not be read - choose another site."

    cands = build_candidates(co, home)
    save_candidates(cands, run.dir / "candidates")
    candidates[co.company_id] = cands
    print(f"  {cands['raw_link_count']} links on the page -> {len(cands['candidates'])} candidates "
          f"({len(cands['dropped'])} dropped, reasons saved)\n")

אייבורי: HTTP 200, final URL https://www.ivory.co.il/, 68,879 chars of text, error: None
  2398 links on the page -> 720 candidates (1678 dropped, reasons saved)

לאסטפרייס: HTTP 200, final URL https://www.lastprice.co.il/, 15,836 chars of text, error: None
  512 links on the page -> 487 candidates (25 dropped, reasons saved)



**4b. Link selection by the model (2 calls), validated in code.** The model picks up to 3 pages, each with a
`type`, `url` and `reason`. The answer is checked before anything is fetched. If anything is wrong, the whole
answer is rejected, its raw output and the reasons are saved, and nothing is fetched from it.
Re-running this cell retries only the rejected companies, and each retry is logged as a new attempt.

In [5]:
for co in companies:
    if selections.get(co.company_id, {}).get("status") == "accepted":
        continue
    sel = select_links(co, candidates[co.company_id], run=run)
    selections[co.company_id] = sel
    print(f"{co.name}: {sel['status']} (attempt {sel['attempt']}, {sel['duration_seconds']}s, tokens {sel['usage']})")
    for link in sel["links"]:
        print(f"  - [{link['type']}] {link['normalized_url']}\n      reason: {link['reason']}")
    for reason in sel["rejection_reasons"]:
        print("  ! rejected:", reason)
    print()

not_accepted = [co.name for co in companies if selections[co.company_id]["status"] != "accepted"]
if not_accepted:
    raise RuntimeError(f"Link selection was not accepted for {not_accepted}: nothing is fetched from it. "
                       "The raw output and reasons are in selections/. Re-run this cell to try again.")

אייבורי: accepted (attempt 1, 3.82s, tokens {'prompt_tokens': 22706, 'completion_tokens': 225, 'total_tokens': 22931})
  - [careers] https://www.ivory.co.il/jobs
      reason: For candidates: lists open positions at Ivory and explains how to apply.
  - [stores] https://www.ivory.co.il/stores.html
      reason: For both: shows Ivory's physical store locations. Customers learn where to shop and pick up orders, and candidates see where branch roles are based.
  - [products] https://www.ivory.co.il/catalog.php?act=all_categories
      reason: For customers: an overview of the full range of product categories, from computers and phones to home appliances, showing the breadth of what Ivory sells.

לאסטפרייס: accepted (attempt 1, 2.97s, tokens {'prompt_tokens': 80280, 'completion_tokens': 179, 'total_tokens': 80459})
  - [about] https://www.lastprice.co.il/About
      reason: Company background and what LastPrice offers; useful for both customers and candidates.
  - [careers] https://www.last

**4c. Source collection.** The home page becomes A1 / B1 and the selected pages follow in the model's order
(A2–A4 / B2–B4). Each source is `ok`, `empty` or `failed`. Empty and failed sources stay in the record so
the gaps are visible, but they are never used as evidence.

In [6]:
for co in companies:
    col = collect_sources(co, homes[co.company_id], selections[co.company_id], run=run)
    save_sources(col, run.dir / "sources")
    collections[co.company_id] = col
    print(summarize_sources(col), "\n")

אייבורי (4 requests to the site)
  A1  ok       68879 chars  https://www.ivory.co.il/
  A2  ok       24298 chars  https://www.ivory.co.il/jobs
  A3  ok       20526 chars  https://www.ivory.co.il/stores.html
  A4  ok       27343 chars  https://www.ivory.co.il/catalog.php?act=all_categories
  usable sources: 4 of 4 

לאסטפרייס (4 requests to the site)
  B1  ok       15836 chars  https://www.lastprice.co.il/
  B2  ok       16543 chars  https://www.lastprice.co.il/About
  B3  ok       18283 chars  https://www.lastprice.co.il/Jobs
  B4  ok       19680 chars  https://www.lastprice.co.il/WhyLP
  usable sources: 4 of 4 



**If a company has too little usable information**, add a replacement page from its saved candidate list.
The replacement takes a free ID within the page limit, and the reason is recorded. Uncomment only if needed:

In [7]:
# co = COMPANY_A
# add_replacement_source(collections[co.company_id], candidates[co.company_id],
#                        "https://...", "why this page is needed", run=run)
# save_sources(collections[co.company_id], run.dir / "sources")
# print(summarize_sources(collections[co.company_id]))

## 5. Contexts

All outputs are built from the **saved** sources, and nothing is fetched again. Only `ok` sources are used, within
a budget of source-text characters (`CONTEXT_CHAR_BUDGET` in `brochure/config.py`).

**Site-wide menu text first.** Many sites repeat the same menu at the top of every page, so the start of each page is mostly
menu. Lines that appear on every usable page of a company are removed from pages 2–4 and moved to the end of the home page.
The page's own text comes first, and the menu stays available in the home page. The saved sources are not changed.

The budget is then split fairly, so a long page is shortened instead of pushing the others out. Instructions, headers and
source IDs are outside the budget and are never cut. For the comparison, both companies share one budget.

In [8]:
saved = {co.company_id: load_sources(run.dir / "sources" / f"{co.company_id}.json") for co in companies}

contexts = {co.company_id: build_context([saved[co.company_id]]) for co in companies}
contexts["comparison"] = build_context([saved[COMPANY_A.company_id], saved[COMPANY_B.company_id]])

for name, ctx in contexts.items():
    print(f"[{name}]")
    print(format_report(ctx), "\n")

[ivory]
Source text used: 11,996 of 12,000 characters (site-wide menu handling: on)
  A1  shortened   7,758 / 68,879  site-wide menu text moved to the end (45,883 chars); shortened to its fair share of the budget (7,762 characters)
  A2  included    3,930 / 24,298  site-wide menu text removed (20,368 chars); fits within its share of the budget
  A3  included      165 / 20,526  site-wide menu text removed (20,361 chars); fits within its share of the budget
  A4  included      143 / 27,343  site-wide menu text removed (27,200 chars); fits within its share of the budget 

[lastprice]
Source text used: 11,794 of 12,000 characters (site-wide menu handling: on)
  B1  shortened   3,342 / 15,836  site-wide menu text moved to the end (14,750 chars); shortened to its fair share of the budget (3,348 characters)
  B2  included    1,954 / 16,543  site-wide menu text removed (14,589 chars); fits within its share of the budget
  B3  shortened   3,258 / 18,283  site-wide menu text removed (14,589 char

## 6. Brochures (4 calls)

The same company facts serve two different readers. Each audience gets its own structure and emphasis.
Each brochure is checked in code right after it is generated: cited IDs, source list, word count (200–300 with `split`)
and required headings. The **first brochure is streamed**.

### 6.1 Company A, potential customers (streamed)

In [9]:
results = {}
results["A_customers"] = generate_brochure(COMPANY_A, "customers", contexts[COMPANY_A.company_id], run, stream=True)

<div dir="rtl">

# אייבורי: מידע ללקוחות פוטנציאליים

## הבעיה שהחברה פותרת
אייבורי מתאימה ללקוחות שמחפשים מקום אחד לקנות מחשבים, סלולר, מוצרי חשמל וגיימינג, באתר או בחנות פיזית. החברה מציגה את עצמה כ"הרשת המובילה למחשבים, סלולר ומוצרי חשמל" [A1].

## מוצרים ושירותים
- **ציוד רשת**: נתבים, מערכות Mesh ומתאמי רשת של TP-Link ו-Cudy, שמאפשרים לשפר את כיסוי ה-Wi-Fi בבית [A1].
- **גיימינג**: משחקים ל-PS5, PS4 ו-Nintendo Switch, ואביזרים כמו הגה Logitech G29 [A1].
- **סלולר ואודיו**: סמארטפונים כמו Samsung Galaxy A07, אוזניות JBL ו-Sennheiser וסוללות גיבוי [A1].
- **מוצרי חשמל לבית**: מכונות קפה, שואבים שוטפים ורובוטיים ומצלמות אבטחה [A1].
- **מבצעים**: באתר מוצגים מבצעי BLACK FRIDAY ונובמבר, עם מחיר מבצע נפרד לאילת [A1].
- **רכישה וייעוץ**: באתר יש חשבון לקוח עם היסטוריית הזמנות וכרטיסי אשראי שמורים [A2][A3]. מודעות הגיוס של החברה מתארות ייעוץ ללקוחות בסניפים, במוקד הטלפוני ובוואטסאפ [A2].

## שימושים וקהלי יעד
המוצרים מיועדים לקהל פרטי שמצטייד בטכנולוגיה לבית ולבידור: שדרוג רשת ביתית, גיימינג, סלולר ומוצרי חשמל [A1]. לדברי החברה, החנויות בפריסה ארצית, באזורים כמו מרכז שרון, דרום שפלה, ירושלים והסביבה וצפון [A3]. התאמה ללקוחות עסקיים: לא נמצא במקורות שנמסרו למודל.

## מה כדאי לברר לפני פנייה
- מהי מדיניות האחריות וההחזרות? לא נמצא במקורות שנמסרו למודל.
- מהן עלויות המשלוח וזמני האספקה? לא נמצא במקורות שנמסרו למודל.
- מהן הכתובת ושעות הפתיחה של הסניף הקרוב? לא נמצא במקורות שנמסרו למודל.
- האם יש ללקוחות שירות הרכבת מחשבים או תיקונים? לא נמצא במקורות שנמסרו למודל.

## מקורות
- [A1] https://www.ivory.co.il/
- [A2] https://www.ivory.co.il/jobs
- [A3] https://www.ivory.co.il/stores.html

</div>

[checks: ivory_customers] PASSED | words=233 (range 200-300) | cited=['A1', 'A2', 'A3'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x5


### 6.2 Company A, job candidates

In [10]:
results["A_candidates"] = generate_brochure(COMPANY_A, "candidates", contexts[COMPANY_A.company_id], run)

<div dir="rtl">

# אייבורי: מידע למועמדים לעבודה

## במה החברה עוסקת
אייבורי היא רשת קמעונאית למחשבים, סלולר ומוצרי חשמל, ובאתר היא מציגה את עצמה כ"הרשת המובילה" בתחום [A1]. העבודה ברשת כוללת מכירה וייעוץ בסניפים ובמוקד טלפוני. יש בה גם הרכבת מחשבים, הובלת סחורה לסניפים וטיפול בהחזרות מול ספקים [A2]. לרשת סניפים באזורים שונים, בהם מרכז שרון, דרום שפלה, ירושלים והסביבה וצפון [A3].

## תפקידים וצוותים שפורסמו
- **מכירות בסניף**: תל אביב (רמת אביב וכללי), רעננה ובית שמש. התפקיד כולל מכירת מחשבים ומוצרי טכנולוגיה וייעוץ ללקוחות [A2].
- **מכירות טלפוניות**: פתח תקווה. מכירה וייעוץ בטלפון ובוואטסאפ [A2].
- **מרכיב/ת מחשבים**: פתח תקווה. נדרשת היכרות עם רכיבי חומרה, וניסיון בהרכבה הוא יתרון משמעותי [A2].
- **רכז/ת RMA**: פתח תקווה. עבודה מול ספקים בארץ ובחו"ל וטיפול בהחלפות, החזרות וזיכויים. נדרשות אנגלית טובה ושליטה ב-Priority [A2].
- **נהגי משאית**: פתח תקווה. נדרש רישיון C1, התפקיד כולל הובלה לסניפים, והעבודה בימים א'-ה' בשעות 07:00–15:00 [A2].

## סביבת העבודה
לדברי החברה, סביבת העבודה "צעירה ודינמית", ובמודעת המוקד הטלפוני היא מתוארת גם כ"משפחתית" [A2]. המודעות מציינות משרה מלאה, אפשרויות קידום והתפתחות ברשת ושכר גבוה, ובתפקידי המכירות גם בונוסים [A2]. דרישות שחוזרות בכמה מודעות: תודעת שירות גבוהה וזיקה לעולם הטכנולוגיה [A2]. עבודה מרחוק: לא נמצא במקורות שנמסרו למודל.

## תהליך המועמדות
שולחים קורות חיים דרך הטופס שבתוך כל משרה בעמוד הדרושים. צירוף קובץ מסוג doc, docx או pdf אינו חובה [A2]. שלבי המיון: לא נמצא במקורות שנמסרו למודל.

שאלות ששווה לשאול בראיון:
- איך נראית ההכשרה המקצועית בתחילת העבודה?
- מה מבנה הבונוסים, ואילו מסלולי קידום קיימים ברשת?

## מקורות
- [A1] https://www.ivory.co.il/
- [A2] https://www.ivory.co.il/jobs
- [A3] https://www.ivory.co.il/stores.html

</div>

[checks: ivory_candidates] PASSED | words=248 (range 200-300) | cited=['A1', 'A2', 'A3'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x2


### 6.3 Company B, potential customers

In [11]:
results["B_customers"] = generate_brochure(COMPANY_B, "customers", contexts[COMPANY_B.company_id], run)

<div dir="rtl">

# לאסטפרייס: מידע ללקוחות פוטנציאליים

## הבעיה שהחברה פותרת
לאסטפרייס מיועדת למי שרוצה לקנות באינטרנט מוצרי חשמל ומוצרים לבית במחיר משתלם, עם אפשרויות משלוח, החזרה ושירות אנושי [B1][B4].

## מוצרים ושירותים
- **חנות מקוונת**: אפשר לרכוש מוצרי חשמל קטנים וגדולים, כמו מקררים, מכונות כביסה, מוצרי מיזוג וסלולר [B1]. יש גם בשמים, רהיטים, כלי עבודה וטקסטיל [B4]. לדברי החברה, עשרות אלפי מוצרים זמינים, והיא מתחייבת למחיר הטוב ביותר [B4].
- **משלוח**: בוחרים את שיטת השילוח בעמוד התשלום: לוקרים של Boxit, שליח עד הבית או המשרד, או הובלה [B4].
- **תשלום**: PayPal או כרטיס אשראי בשרת בתקן PCI. לדברי החברה, פרטי האשראי אינם נשמרים אצלה [B4].
- **החזרות**: אפשר להחזיר מוצר תוך 14 יום לפי תקנון האתר וחוק הגנת הצרכן. ההחזר מלא, למעט דמי שינוע ואריזה. ההחזרה לא חלה על רהיטים, בשמים, מוצרי ספורט ומזון [B4].
- **שירות**: מוקד טלפוני 1-700-70-80-89, נציג ב-WhatsApp וסניפים [B1]. שינויים בהזמנה נעשים דרך המוקד או דרך תיק הלקוח באתר [B4].
- **LastPrice Plus**: לקוחות המסלול זכאים לזכות החזרה מורחבת [B4].

## שימושים וקהלי יעד
- צרכנים פרטיים שמצטיידים לבית ולמטבח. באתר יש גם מגזין עם מתכונים ומדריכי מוצר [B1].
- עסקים: קיים מסלול "לאסטפרייס לעסקים" [B1][B4], אך פרטיו: לא נמצא במקורות שנמסרו למודל.
- החברה מציינת שהאתר שירת יותר מ-500,000 לקוחות [B2].

## מה כדאי לברר לפני פנייה
- מי נותן את האחריות והשירות הטכני למוצר שמעניין אתכם? לאסטפרייס אינה יבואן [B4]. פרטי היבואן לכל מוצר: לא נמצא במקורות שנמסרו למודל.
- כמה עולה LastPrice Plus ומה הוא כולל? לא נמצא במקורות שנמסרו למודל.
- מהם זמני המשלוח ועלויותיו לאזור שלכם? לא נמצא במקורות שנמסרו למודל.
- מהם תנאי המחיר וההתקשרות לעסקים? לא נמצא במקורות שנמסרו למודל.

## מקורות
- [B1] https://www.lastprice.co.il/
- [B2] https://www.lastprice.co.il/About
- [B4] https://www.lastprice.co.il/WhyLP

</div>

[checks: lastprice_customers] PASSED | words=271 (range 200-300) | cited=['B1', 'B2', 'B4'] of included=['B1', 'B2', 'B3', 'B4'] | missing-info phrase x5


### 6.4 Company B, job candidates

In [12]:
results["B_candidates"] = generate_brochure(COMPANY_B, "candidates", contexts[COMPANY_B.company_id], run)

<div dir="rtl">

# לאסטפרייס: מידע למועמדים לעבודה

## במה החברה עוסקת
לאסטפרייס הוא אתר מסחר מקוון למוצרי צריכה, ובעיקר למוצרי חשמל ביתיים. האתר נוסד ב-2002 ומופעל כיום על ידי קבוצת ניופאן וידיעות אינטרנט [B2]. העבודה כוללת מכירה ושירות, קניינות, ניהול מלאי והזמנות, תוכן ודיגיטל ופיתוח. לדברי החברה, יש לה מרכז לוגיסטי עם עשרות אלפי פריטים [B2].

## תפקידים וצוותים שפורסמו
בעמוד הצוות מופיעים הצוותים הבאים: הנהלה, כספים, הזמנות ובק אופיס, מכירות, תוכן ודיגיטל, סחר, סניפים (אילת) וצוות TOTAL [B2].

משרות שפורסמו [B3]:
- **נציג/ת מכירות ושירות טלפוני** (#105)
- **נציג/ת מכירות בסניף חיפה** (#108): טיפול בהזמנות, הצעות מחיר ולקוחות עסקיים
- **מנהל/ת חשבונות** (#101): נדרשים Excel ו-ERP
- **נציג/ת תוכן מסחרי** (#106): העלאת תכנים לאתר. אנגלית ופוטושופ הם יתרון
- **מתכנת/ת בסביבת Windows** (#111): .NET, MSSQL וממשקי API
- **פקיד/ת הזמנות**

ברוב המשרות נדרשת תעודת בגרות מלאה [B3].

## סביבת העבודה
- **מיקום:** לפי החברה, כל המשרות במיקום מרכזי בחיפה, בסביבה "ממוזגת, צעירה וממוחשבת" [B3].
- **נגישות:** המשרד קרוב למטרונית, לכרמלית ולרכבת [B3].
- **היקף משרה:** המשרות שפורסמו הן במשרה מלאה [B3].
- **שכר וקידום:** החברה מציינת "שכר הולם" ובונוסים. למשרת המכירות הטלפונית היא מציינת אפשרות קידום לתוכן, לניהול ולשיווק [B3].
- **עבודה מרחוק:** לא נמצא במקורות שנמסרו למודל.

## תהליך המועמדות
- **אופן הגשה:** הגשת קורות חיים במייל [B3].
- **כתובת המייל:** לא נמצא במקורות שנמסרו למודל.
- **שלבי התהליך:** לא נמצא במקורות שנמסרו למודל.
- **המלצות:** למשרת המתכנת ייתכן שיידרשו המלצות [B3].

שאלות שכדאי לשאול בראיון:
- מה כולל מסלול הקידום בפועל, ותוך כמה זמן?
- איך מחושבים הבונוסים?

## מקורות
- [B2] https://www.lastprice.co.il/About
- [B3] https://www.lastprice.co.il/Jobs

</div>

[checks: lastprice_candidates] PASSED | words=249 (range 200-300) | cited=['B2', 'B3'] of included=['B1', 'B2', 'B3', 'B4'] | missing-info phrase x3


## 7. Comparison (1 call)

Four dimensions fixed before generation (`COMPARISON_DIMENSIONS` in `brochure/config.py`): products and services,
target audiences, pricing or trial, and working at the company. It is based on the companies' own publications, so it
is not independent market research.

In [13]:
results["comparison"] = generate_comparison(COMPANY_A, COMPANY_B, contexts["comparison"], run)

<div dir="rtl">

# השוואה: אייבורי מול לאסטפרייס

| ממד | אייבורי | לאסטפרייס |
|---|---|---|
| מוצרים ושירותים | לפי כותרת האתר, רשת למחשבים, סלולר ומוצרי חשמל [A1]. בדף הבית מוצגים, למשל, נתבים, אוזניות, משחקי PS5, שואב שוטף ומחמם מים [A1]. לחברה חנויות בפריסה ארצית, באזורים כמו מרכז שרון, דרום שפלה, ירושלים והסביבה וצפון [A3]. | לדברי החברה, אתר מסחר מקוון שנוסד ב-2002 ומתמקד במיוחד במוצרי חשמל ביתיים [B2]. האתר מציע גם בשמים, רהיטים, כלי עבודה וטקסטיל [B4]. בנוסף יש מגזין תוכן [B1][B4] וסניף בחיפה [B3]. |
| קהלי יעד | לא נמצא במקורות שנמסרו למודל | קיים מסלול "לאסטפרייס לעסקים" [B1][B4] ומנוי LastPrice Plus [B4]. לדברי החברה, האתר שירת מעל 500,000 לקוחות [B2]. |
| תמחור או תקופת ניסיון | מוצגים מחיר רגיל, מחיר מבצע ומחיר מבצע באילת, במסגרת מבצעי BLACK FRIDAY ו"מבצע נובמבר SALE" [A1]. תקופת ניסיון: לא נמצא במקורות שנמסרו למודל | מוצגים מחירים ב-₪ ב"המוצרים החמים" [B1]. החברה מציינת "התחייבות למחיר הטוב ביותר". לקוחות Plus זכאים, לדבריה, לזכות החזרה מורחבת, בכפוף לתנאי הכוכבית [B4]. תקופת ניסיון: לא נמצא במקורות שנמסרו למודל |
| עבודה בחברה | מוצגות משרות למכירות בסניף (תל אביב, רעננה), מרכיב/ת מחשבים, רכז/ת RMA ונהגי משאית בפתח תקווה [A2]. לפי המודעות: שכר גבוה ובונוסים, אופציות קידום ומשרה מלאה. לנהגים: א'-ה' בשעות 07:00–15:00 [A2]. | לפי החברה, כל המשרות בחיפה [B3]. מוצגות משרות לנציגי מכירות ושירות טלפוני, נציג/ת מכירות בסניף ומנהל/ת חשבונות. נדרשת תעודת בגרות מלאה במשרות המפורטות. מוצעים שכר הולם ובונוסים, במשרה מלאה [B3]. |

## הבדלים עיקריים
- **מיקום המשרות:** אייבורי מפרסמת משרות בתל אביב, ברעננה ובפתח תקווה [A2]. לאסטפרייס מציינת שכל משרותיה בחיפה [B3]. במשרות המפורטות של לאסטפרייס נדרשת בגרות מלאה [B3]. דרישה כזו לא מופיעה במודעות של אייבורי שנמסרו [A2].
- **אופן המכירה:** אייבורי מציגה רשת חנויות ארצית [A3]. לאסטפרייס מציגה את עצמה בעיקר כאתר מקוון עם סניף בחיפה [B2][B3].
- **תמחור:** אצל אייבורי המחירים המוצגים מתויגים כמבצעים מוגבלים בזמן (BLACK FRIDAY, נובמבר), עם מחיר נפרד לאילת [A1]. תנאי המבצעים ומשכם אינם מפורטים במקור. אצל לאסטפרייס יש בעיקר טענה כללית על המחיר הטוב ביותר ומנוי Plus שתנאיו אינם מפורטים במקורות [B4]. לכן קשה להשוות מחירים ישירות.
- **קהלי יעד:** לא ניתן להשוות, כי מקורות אייבורי שנמסרו אינם מגדירים קהל יעד.

## למי כל חברה עשויה להתאים
אייבורי עשויה להתאים למי שמחפש מוצרי מחשוב, רשת וגיימינג וגם רכישה בחנות פיזית [A1][A3]. לאסטפרייס עשויה להתאים למי שמחפש מגוון רחב של מוצרי חשמל ביתיים ומוצרים נוספים, או רכישה לעסקים [B4]. מחפשי עבודה צריכים לשקול בעיקר את מיקום המשרות. לפני החלטה כדאי לבדוק תנאי משלוח והחזרה, את תנאי מנוי Plus, את תוקף המבצעים ואם המשרות עדיין פתוחות.

ההשוואה מבוססת על פרסומי החברות עצמן ואינה מחקר שוק בלתי תלוי, ופירוט רב יותר באתר של חברה אחת אינו מעיד שהיא טובה יותר.

## מקורות
- [A1] https://www.ivory.co.il/
- [A2] https://www.ivory.co.il/jobs
- [A3] https://www.ivory.co.il/stores.html
- [B1] https://www.lastprice.co.il/
- [B2] https://www.lastprice.co.il/About
- [B3] https://www.lastprice.co.il/Jobs
- [B4] https://www.lastprice.co.il/WhyLP

</div>

[checks: comparison] PASSED | cited=['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'B4'] of included=['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4'] | missing-info phrase x3


## 8. Baseline brochure for the prompt experiment (1 call)

Same company, audience, model, settings and **exact same context** as brochure 6.1, with the same language, length and
source / citation rules. Only the prompt wording and the requested structure change (the class notebook's simple prompt).

In [14]:
results["A_customers_baseline"] = generate_brochure(
    COMPANY_A, "customers", contexts[COMPANY_A.company_id], run, variant="baseline")

<div dir="rtl">

# אייבורי: מחשבים, סלולר ומוצרי חשמל

## מי אנחנו
אייבורי מציגה את עצמה כ"הרשת המובילה למחשבים, סלולר ומוצרי חשמל" ומבטיחה "חוויה ייחודית של טכנולוגיה" [A1]. לדברי החברה, החנויות פרוסות בפריסה ארצית [A3].

## מה אפשר לקנות
- ציוד רשת: נתבים ומגדילי טווח של TP-Link ו-Cudy [A1].
- גיימינג: משחקים ל-PS5, ל-PS4 ול-Nintendo Switch 2, והגה Logitech G29 [A1].
- אודיו: אוזניות JBL ו-Sennheiser ורמקולי Edifier [A1].
- סלולר, כגון Samsung Galaxy A07 [A1].
- מוצרי חשמל לבית: מכונות קפה, שואבים של Bissell ו-Roborock ומחממי מים [A1].
- צילום ואבטחה: גימבלים של DJI, מצלמות אבטחה ביתיות ומצלמות אקסטרים [A1].
- בקטלוג המוצרים מופיעה גם קטגוריית מצלמות רשת ואינטרנט [A4].
- בשמים ומוצרים במותג Ivory [A1].

## מבצעים ומחירים
באתר מוצגים מבצעי BLACK FRIDAY ו"מבצע נובמבר SALE". לצד מחיר המבצע מופיע לרוב מחיר מבצע נפרד ונמוך יותר לאילת [A1]. לדוגמה, אוזניות Sennheiser MOMENTUM 5 Wireless ANC מוצגות במסגרת BLACK FRIDAY [A1].

## סניפים ושירות
- הסניפים מחולקים לאזורים: מרכז שרון, דרום שפלה, ירושלים והסביבה וצפון [A3].
- מודעות הדרושים מזכירות סניפים בתל אביב (רמת אביב), ברעננה ובבית שמש [A2].
- לפי החברה, אנשי המכירות בסניפים מעניקים ללקוחות שירות וייעוץ [A2].
- מוקד מכירות טלפוני בפתח תקווה מטפל בפניות בטלפון ובוואטסאפ ומספק ייעוץ מקצועי [A2].
- החברה מגייסת מרכיבי מחשבים ורכז RMA לטיפול בהחלפות, החזרות וזיכויים מול ספקים [A2].
- החברה מגייסת נהגי משאית להובלת סחורה לסניפים ברחבי הארץ [A2].

## מידע נוסף
- אחריות, משלוחים ושעות פתיחה: לא נמצא במקורות שנמסרו למודל.
- כתובות הסניפים ופרטי התקשרות: לא נמצא במקורות שנמסרו למודל.
- שירות תיקונים ומעבדה: לא נמצא במקורות שנמסרו למודל.

## מקורות
- [A1] https://www.ivory.co.il/
- [A2] https://www.ivory.co.il/jobs
- [A3] https://www.ivory.co.il/stores.html
- [A4] https://www.ivory.co.il/catalog.php?act=all_categories

</div>

[checks: ivory_customers_baseline] PASSED | words=259 (range 200-300) | cited=['A1', 'A2', 'A3', 'A4'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x3


## 9. Run summary

In [15]:
rows = ["| Output | Run ID | Status | Words | Checks |", "|---|---|---|---|---|"]
for r in results.values():
    words = r["word_count"] if r["word_count"] is not None else "-"
    checks = "passed" if r["checks"]["passed"] else "; ".join(r["checks"]["problems"])
    rows.append(f"| {r['name']} | {r['run_id']} | {r['status']} | {words} | {checks} |")
display(Markdown("\n".join(rows)))

print(run.format_summary())
run.save_summary()

| Output | Run ID | Status | Words | Checks |
|---|---|---|---|---|
| ivory_customers | 20261128-095839-d673 | complete | 233 | passed |
| ivory_candidates | 20261128-095839-d673 | complete | 248 | passed |
| lastprice_customers | 20261128-095839-d673 | complete | 271 | passed |
| lastprice_candidates | 20261128-095839-d673 | complete | 249 | passed |
| comparison | 20261128-095839-d673 | complete | - | passed |
| ivory_customers_baseline | 20261128-095839-d673 | complete | 259 | passed |

Run 20261128-095839-d673
  model calls: 8 (planned 8) by action {'select_links': 2, 'brochure': 5, 'comparison': 1}; failed 0; repeated attempts 0; total 149.71s
  tokens (reported by the API, 8 calls): {'prompt_tokens': 163280, 'completion_tokens': 17089, 'total_tokens': 180369}
  requests to company sites: 8 {'ivory': 4, 'lastprice': 4}; failed 0


WindowsPath('C:/Users/User/Documents/cursor/LLL_HM/LLM_HW1_Brochure_Maker/runs/20261128-095839-d673/summary.json')

If a brochure fails a check (for example, the word count), keep the original output and either document it or
regenerate it. A regeneration is logged as attempt 2, saved as `<name>_attempt2`, and overwrites nothing.

## 10. Replay from saved files (no model calls)

Re-runs the checks on every saved output, and rebuilds every context from the saved sources to show it matches the
context that was actually sent. Set `REPLAY_RUN_ID` to any earlier run ID to re-check that run without calling the model.

In [14]:
REPLAY_RUN_ID = run.run_id          # or an earlier run, e.g. "20261127-120000-abcd"
replay = RunLog(REPLAY_RUN_ID)

import json
from brochure.context import read_text_exact, rebuild_context

saved_sources = {p.stem: load_sources(p) for p in (replay.dir / "sources").glob("*.json")}
for meta_path in sorted((replay.dir / "outputs").glob("*.json")):
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    name = meta_path.stem
    print(f"{name}: {format_checks(check_saved_output(replay.dir, name))}")

    # Rebuild with the same settings the saved context was built with (budget, site-wide menu handling),
    # and compare with the exact copy of the context that was sent (kept in contexts/<name>.json).
    ids = meta["company_id"].split("+")              # "a" for a brochure, "a+b" for the comparison
    saved_ctx = json.loads((replay.dir / "contexts" / f"{name}.json").read_text(encoding="utf-8"))
    rebuilt = rebuild_context(saved_ctx, [saved_sources[i] for i in ids])
    print(f"   context rebuilt from saved sources == context sent: {rebuilt['text'] == saved_ctx['text']}")

    # The .txt file is a readable copy of the same text. Runs saved before the line-ending fix wrote it with
    # Windows newline conversion, so for those runs it can differ in line endings only (the .json copy is exact).
    txt_exact = read_text_exact(replay.dir / "contexts" / f"{name}.txt") == saved_ctx["text"]
    print(f"   .txt copy is byte-exact: {txt_exact}")

comparison: PASSED | cited=['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'B4'] of included=['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4'] | missing-info phrase x3
   context rebuilt from saved sources == context sent: True
   .txt copy is byte-exact: False
ivory_candidates: PASSED | words=248 (range 200-300) | cited=['A1', 'A2', 'A3'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x2
   context rebuilt from saved sources == context sent: True
   .txt copy is byte-exact: False
ivory_customers: PASSED | words=233 (range 200-300) | cited=['A1', 'A2', 'A3'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x5
   context rebuilt from saved sources == context sent: True
   .txt copy is byte-exact: False
ivory_customers_baseline: PASSED | words=259 (range 200-300) | cited=['A1', 'A2', 'A3', 'A4'] of included=['A1', 'A2', 'A3', 'A4'] | missing-info phrase x3
   context rebuilt from saved sources == context sent: True
   .txt copy is byte-exact: False
lastprice_candidates: PASSED 

## 11. Manual claim audit (8 claims)

**Sampling rule (fixed in advance):** in each of the 4 improved brochures, take the **first two factual claims** in
reading order. A factual claim is a sentence or bullet in the body with at least one citation that is not a
"לא נמצא במקורות שנמסרו למודל" statement. Headings, the source list and table rows are not claims.
If a brochure has fewer than two, that is recorded as a gap.

The cell below applies the rule to the saved brochures and shows lines from the cited sources that share words with each
claim. That is only a reading aid: the verdict comes from reading the source. A cited ID existing is **not** proof that the
source supports the claim.

In [8]:
AUDIT_NAMES = [f"{co.company_id}_{aud}" for co in companies for aud in ("customers", "candidates")]
claims = audit.sample_claims(replay.dir, AUDIT_NAMES, per_brochure=2)
print(audit.format_claims(claims))

audit_path = replay.dir / "audit" / "claim_audit.csv"
if not audit_path.exists():
    audit.write_audit_template(claims, audit_path)
print("Fill in evidence / verdict / note in:", audit_path)
print("Verdicts:", " | ".join(audit.VERDICTS))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\User\\Documents\\cursor\\LLL_HM\\LLM_HW1_Brochure_Maker\\runs\\20261128-101549-d492\\outputs\\ivory_customers.md'

**Audit results (fill in):**

| # | Brochure | Claim (as written) | Source ID | URL | Evidence quote from the source | Verdict | Correction or explanation |
|---|---|---|---|---|---|---|---|
| 1 | | | | | | | |
| 2 | | | | | | | |
| 3 | | | | | | | |
| 4 | | | | | | | |
| 5 | | | | | | | |
| 6 | | | | | | | |
| 7 | | | | | | | |
| 8 | | | | | | | |

The original outputs stay unchanged in `outputs/`. Any correction is written here.

## 12. Prompt experiment: baseline vs improved

**Criteria, fixed before reading the outputs:**
1. **Faithfulness:** every claim is supported by the cited source, with nothing invented and marketing claims attributed.
2. **Audience fit:** structure, order and explanations serve a potential customer.
3. **Information coverage:** problem, offering, uses / audiences, and open questions.
4. **Clarity:** easy to scan, with clear headings and no filler.

The same model, settings, context, audience, language, length and source rules were used for both. Only the
prompt wording and the requested structure differ. This is one small comparison, not a general ranking of prompts or models.

In [15]:
improved, baseline = results["A_customers"], results["A_customers_baseline"]
for label, r in (("Improved", improved), ("Baseline", baseline)):
    print(f"{label}: {r['word_count']} words, cited {r['checks']['cited_ids']}, "
          f"'not found' x{r['checks']['missing_info_mentions']}, checks {'passed' if r['checks']['passed'] else r['checks']['problems']}")

def column(title, text):
    return (f'<div style="flex:1; min-width:300px; border:1px solid #ccc; border-radius:8px; padding:8px 16px">'
            f'<h3 dir="ltr">{title}</h3>\n\n{rtl_markdown(text).data}\n\n</div>')

display(Markdown('<div style="display:flex; gap:16px; flex-wrap:wrap">\n\n'
                 + column("Improved", improved["text"]) + column("Baseline", baseline["text"]) + "\n\n</div>"))

NameError: name 'results' is not defined

**Findings (fill in, with concrete examples quoted from both versions):**

| Criterion | Improved | Baseline | Example |
|---|---|---|---|
| Faithfulness | | | |
| Audience fit | | | |
| Information coverage | | | |
| Clarity | | | |

Where the improved prompt did **not** help, say so too.

## 13. Discussion (to complete)

- **Code vs model:** what the code does (links, validation, fetching, budget, checks, logging) and what the model does
  (choosing pages, writing).
- **Context management:** the character budget, fair split, what was shortened or left out in this run.
- **Validation points:** link selection, source status, citations / source list / word count, and the manual audit.
- **Changing the prompt vs changing the source material:** how each can change the output.
- **Workflow vs agent:** this is a fixed workflow. What would be needed for an agent that chooses tools at run time.